# 03a — Groundsource: Extract IMERG Precipitation Matrices

**Purpose:** For every flood event, download a 3-D precipitation array from NASA IMERG V07
via Google Earth Engine. The array covers 72 hours before to 24 hours after the event start,
at 0.1° spatial resolution. A binary raster mask of the flood polygon is also created so that
later notebooks can compute spatially-averaged rainfall over the flooded area.

**IMERG availability:** The IMERG V07 'final' run starts on 2000-06-01. Events before this
date are excluded. When the 'final' product is unavailable for a given period, the script
falls back to the IMERG 'late' run.

**Method:** Events are processed in batches of 1,000. Batch files are saved as Parquet to allow
resuming. A progress counter and elapsed time are printed per batch.

**Input:**
- `outputs/groundsource_df_with_pfdi.parquet` — flood events with PFDI metrics (from 02a)

**Output:**
- `outputs/imerg_batches/imerg_batch_*.parquet` — intermediate per-batch results
- `outputs/groundsource_with_imerg.parquet` — all events with precipitation columns:
  - `imerg_matrix` : 3-D numpy array (time × lat × lon) of 30-min rainfall [mm/hr]
  - `imerg_mask`   : 2-D binary array (lat × lon) — 1 inside polygon, 0 outside
  - `imerg_meta`   : dict with origin, scale, timestamps, and array shape
  - `imerg_type`   : 'final' or 'late' (which IMERG product was used)

In [ ]:
import os
import glob
import time
import pickle
import warnings
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import geopandas as gpd
import ee
from shapely import wkb as shapely_wkb

warnings.filterwarnings('ignore')

In [ ]:
# --- CONFIGURATION ---

INPUT_PATH    = r"D:\MY_CODES\global_urban_flood_database\groundsource\outputs\groundsource_df_with_pfdi.parquet"
BATCH_DIR     = r"D:\MY_CODES\global_urban_flood_database\groundsource\outputs\imerg_batches"
OUTPUT_PATH   = r"D:\MY_CODES\global_urban_flood_database\groundsource\outputs\groundsource_with_imerg.parquet"

GEE_PROJECT   = 'your-gee-project'    # Replace with your GEE project ID
IMERG_START   = '2000-06-01'          # earliest date of IMERG V07 availability
HOURS_BEFORE  = 72                    # hours of rainfall to extract before event start
HOURS_AFTER   = 24                    # hours of rainfall to extract after event start
SCALE_DEG     = 0.1                   # spatial resolution of IMERG (degrees)
BATCH_SIZE    = 1_000                 # events per batch file
N_BATCHES_TO_RUN = 900                # batches to process this session; re-run to continue
WGS84_CRS     = "EPSG:4326"

## 1. Initialise GEE and load data

In [ ]:
ee.Initialize(project=GEE_PROJECT)
print("Google Earth Engine initialised.")
os.makedirs(BATCH_DIR, exist_ok=True)

df = pd.read_parquet(INPUT_PATH)
df = df.reset_index(drop=True)
df['event_id']   = df.index
df['start_date'] = pd.to_datetime(df['start_date'])
df['end_date']   = pd.to_datetime(df['end_date'])

# Filter to IMERG availability window
imerg_cutoff = pd.Timestamp(IMERG_START)
df = df[df['start_date'] >= imerg_cutoff].copy()
print(f"Events after IMERG cutoff ({IMERG_START}): {len(df):,}")

## 2. Helper functions

In [ ]:
def get_bounding_box(wkb_bytes):
    geom = shapely_wkb.loads(wkb_bytes)
    minx, miny, maxx, maxy = geom.bounds
    buf = SCALE_DEG
    return minx - buf, miny - buf, maxx + buf, maxy + buf


def rasterize_polygon(wkb_bytes, origin_lon, origin_lat, n_lon, n_lat):
    import rasterio
    from rasterio.transform import from_origin
    from rasterio.features import rasterize
    geom      = shapely_wkb.loads(wkb_bytes)
    transform = from_origin(origin_lon, origin_lat + n_lat * SCALE_DEG, SCALE_DEG, SCALE_DEG)
    return rasterize(
        [(geom.__geo_interface__, 1)],
        out_shape=(n_lat, n_lon),
        transform=transform,
        fill=0, dtype='uint8', all_touched=True
    )


def build_time_window(start_date):
    t_start = start_date - timedelta(hours=HOURS_BEFORE)
    t_end   = start_date + timedelta(hours=HOURS_AFTER)
    return t_start.strftime('%Y-%m-%dT%H:%M:%S'), t_end.strftime('%Y-%m-%dT%H:%M:%S')


def extract_imerg_for_event(wkb_bytes, start_date, product='final'):
    """
    Download a 3-D IMERG precipitation array for one event using toBands()
    (single GEE call per event — same approach as chronicle pipeline).

    Collection IDs and band name match V07:
        final : NASA/GPM_L3/IMERG_V07
        late  : NASA/GPM_L3/IMERG_V07_LATE
        band  : 'precipitation'
    """
    collection_id = (
        'NASA/GPM_L3/IMERG_V07' if product == 'final'
        else 'NASA/GPM_L3/IMERG_V07_LATE'
    )
    t_start_str, t_end_str = build_time_window(start_date)
    min_lon, min_lat, max_lon, max_lat = get_bounding_box(wkb_bytes)
    region = ee.Geometry.BBox(min_lon, min_lat, max_lon, max_lat)

    ic = (ee.ImageCollection(collection_id)
          .filterDate(t_start_str, t_end_str)
          .filterBounds(region)
          .select('precipitation'))

    n_images = ic.size().getInfo()
    if n_images == 0:
        return None, None, None

    # Download all timesteps in a single GEE call
    pixel_dict = ic.toBands().sampleRectangle(region=region, defaultValue=0).getInfo()
    properties = pixel_dict['properties']
    band_keys  = sorted(properties.keys())

    arrays = [np.array(properties[b], dtype=np.float32) for b in band_keys]
    matrix = np.stack(arrays, axis=0)   # shape: (T, H, W)

    n_lat, n_lon = matrix.shape[1], matrix.shape[2]
    origin_lon   = min_lon
    origin_lat   = max_lat

    mask = rasterize_polygon(wkb_bytes, origin_lon, origin_lat, n_lon, n_lat)
    meta = {
        'origin_lon' : origin_lon,
        'origin_lat' : origin_lat,
        'scale_deg'  : SCALE_DEG,
        'shape'      : matrix.shape,
        'timestamps' : band_keys,
    }
    return matrix, mask, meta


def get_completed_event_ids(batch_dir):
    files = glob.glob(os.path.join(batch_dir, 'imerg_batch_*.parquet'))
    if not files:
        return set()
    dfs = [pd.read_parquet(f, columns=['event_id']) for f in files]
    return set(pd.concat(dfs)['event_id'].tolist())

## 3. Batch processing loop

Re-run this cell to continue from where the previous session stopped.

In [ ]:
completed_ids = get_completed_event_ids(BATCH_DIR)
remaining     = df[~df['event_id'].isin(completed_ids)].copy()
print(f"Already processed: {len(completed_ids):,}")
print(f"Remaining         : {len(remaining):,}")

batches_run   = 0
t_session     = time.time()

for batch_start in range(0, len(remaining), BATCH_SIZE):
    if batches_run >= N_BATCHES_TO_RUN:
        print(f"\nSession limit of {N_BATCHES_TO_RUN} batches reached. Re-run to continue.")
        break

    batch      = remaining.iloc[batch_start : batch_start + BATCH_SIZE]
    id_min     = int(batch['event_id'].min())
    id_max     = int(batch['event_id'].max())
    out_file   = os.path.join(BATCH_DIR, f'imerg_batch_{id_min}_{id_max}.parquet')

    if os.path.exists(out_file):
        batches_run += 1
        continue

    t0      = time.time()
    records = []

    for _, row in batch.iterrows():
        matrix = mask = meta = imerg_type = None

        for product in ('final', 'late'):
            try:
                matrix, mask, meta = extract_imerg_for_event(
                    row['geometry'], row['start_date'], product=product
                )
                if matrix is not None:
                    imerg_type = product
                    break
            except Exception:
                continue

        if matrix is None:
            continue   # skip events where no IMERG data is available

        records.append({
            'event_id'    : int(row['event_id']),
            'imerg_matrix': pickle.dumps(matrix),    # serialise array as bytes for Parquet
            'imerg_mask'  : pickle.dumps(mask),
            'imerg_meta'  : pickle.dumps(meta),
            'imerg_type'  : imerg_type,
        })

    if records:
        pd.DataFrame(records).to_parquet(out_file, index=False)

    batches_run += 1
    pct = (len(completed_ids) + batch_start + len(batch)) / len(df) * 100
    print(f"Batch {batches_run}/{N_BATCHES_TO_RUN}  "
          f"| events {id_min}–{id_max}  "
          f"| saved {len(records)}  "
          f"| {time.time()-t0:.1f}s  "
          f"| overall {pct:.1f}%", end='\r')

print(f"\nSession complete. Batches: {batches_run}, elapsed: {time.time()-t_session:.0f}s")

## 4. Merge all batches and save

In [ ]:
batch_files = sorted(glob.glob(os.path.join(BATCH_DIR, 'imerg_batch_*.parquet')))
print(f"Found {len(batch_files)} batch files")

imerg_df = pd.concat([pd.read_parquet(f) for f in batch_files], ignore_index=True)
print(f"Total IMERG records: {len(imerg_df):,}")

# Merge IMERG columns onto the main dataset
merged = df.merge(imerg_df, on='event_id', how='inner')

# Report coverage
skipped = len(df) - len(merged)
print(f"Merged dataset  : {len(merged):,}  (skipped {skipped:,} events with no IMERG data)")
if imerg_df['imerg_type'].notna().any():
    print("IMERG type breakdown:")
    print(merged['imerg_type'].value_counts())

merged.to_parquet(OUTPUT_PATH, index=False)
print(f"Saved → {OUTPUT_PATH}")